# TUGAS MANDIRI — PERTEMUAN 10
## Pemrosesan Data Streaming dengan Spark Structured Streaming (Kopi Nusantara)

- **Nama:** Emannuel Henji
- **NPM:** 2505060012
- **Mata Kuliah:** Praktikum Big Data

---
## A. Menyiapkan Streaming Source

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

# Menyiapkan SparkSession dan folder simulasi streaming
spark = SparkSession.builder.appName("Tugas10").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

skema_tugas = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])
print("Siap untuk tugas. Folder:", folder_tugas)

26/11/11 22:46:55 WARN Utils: Your hostname, mazedev-1701 resolves to a loopback address: 127.0.1.1; using 192.168.100.6 instead (on interface wlp2s0)
26/11/11 22:46:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/11 22:46:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/11 22:46:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Siap untuk tugas. Folder: tugas10_stream


In [2]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming DataFrame berhasil didefinisikan.")
print("df_stream.isStreaming:", df_stream.isStreaming)

Streaming DataFrame berhasil didefinisikan.
df_stream.isStreaming: True


---
## B. Agregasi per Cabang

In [3]:
# Tambahkan kolom turunan total_harga (qty x harga_satuan)
df_stream_total = df_stream.withColumn(
    "total_harga", col("qty") * col("harga_satuan")
)

# Agregasi total_harga dan jumlah pesanan per cabang
agregasi_stream = df_stream_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Jalankan Query dengan memory sink
query = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query 1 (agregasi_cabang) dimulai! Status:", query.status)

Streaming query 1 (agregasi_cabang) dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


---
## C. Agregasi Berbasis Window Waktu

In [4]:
# Definisikan ulang readStream dari folder yang sama untuk streaming query kedua
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas)

df_stream_waktu_total = df_stream_waktu.withColumn(
    "total_harga", col("qty") * col("harga_satuan")
)

# Agregasi total_harga berdasarkan window waktu 2 menit dari kolom waktu_pesan
agregasi_window = df_stream_waktu_total.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query 2 (agregasi_waktu) dimulai! Status:", query2.status)

Streaming query 2 (agregasi_waktu) dimulai! Status: {'message': 'Getting offsets from FileStreamSource[file:/mnt/MazedevUbuntu/tugas/praktikum/big-data/pertemuan-10/tugas10_stream]', 'isDataAvailable': False, 'isTriggerActive': True}


---
## D. Menguji dengan Data Simulasi

In [5]:
# Batch 1 dikirim
pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Susu Gula Aren", "Americano", "Matcha Latte"],
    "qty": [2, 1, 3],
    "harga_satuan": [22000, 18000, 25000],
    "waktu_pesan": ["2026-11-11 10:00:15", "2026-11-11 10:00:40", "2026-11-11 10:01:05"],
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(7)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     119000|             2|
| Yogyakarta|      18000|             1|
+-----------+-----------+--------------+



In [6]:
# Batch 2 dikirim
pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Yogyakarta", "Solo", "Magelang"],
    "nama_menu": ["Espresso", "Cappuccino", "Kopi Tubruk"],
    "qty": [2, 2, 1],
    "harga_satuan": [15000, 24000, 12000],
    "waktu_pesan": ["2026-11-11 10:01:30", "2026-11-11 10:01:45", "2026-11-11 10:01:55"],
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

Batch 2 dikirim. Menunggu Spark memproses...


+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     131000|             3|
| Yogyakarta|      48000|             2|
|       Solo|      48000|             1|
+-----------+-----------+--------------+



In [7]:
# Batch 3 dikirim
pd.DataFrame({
    "pesanan_id": ["P007", "P008", "P009"],
    "nama_cabang": ["Solo", "Semarang", "Yogyakarta"],
    "nama_menu": ["Latte", "Kopi Susu Gula Aren", "Caramel Macchiato"],
    "qty": [3, 4, 2],
    "harga_satuan": [23000, 22000, 28000],
    "waktu_pesan": ["2026-11-11 10:02:10", "2026-11-11 10:02:35", "2026-11-11 10:03:15"],
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

Batch 3 dikirim. Menunggu Spark memproses...


+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     131000|             3|
|       Solo|     117000|             2|
| Yogyakarta|     104000|             3|
|   Semarang|      88000|             1|
+-----------+-----------+--------------+



In [8]:
# Memverifikasi hasil agregasi berbasis window waktu 2 menit (Bagian C)
spark.sql("""
    SELECT window.start, window.end, total_harga, jumlah_pesanan
    FROM agregasi_waktu
    ORDER BY window.start
""").show(truncate=False)

+-------------------+-------------------+-----------+--------------+
|start              |end                |total_harga|jumlah_pesanan|
+-------------------+-------------------+-----------+--------------+
|2026-11-11 10:00:00|2026-11-11 10:02:00|227000     |6             |
|2026-11-11 10:02:00|2026-11-11 10:04:00|213000     |3             |
+-------------------+-------------------+-----------+--------------+



---
## E. Sink Berbasis Berkas (Parquet)

In [9]:
# Streaming query ketiga: menulis data mentah dari df_stream ke format Parquet
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming query 3 (Parquet sink) dimulai! Status:", query3.status)

Streaming query 3 (Parquet sink) dimulai! Status: {'message': 'Initializing StreamExecution', 'isDataAvailable': False, 'isTriggerActive': False}


In [10]:
# Kirim satu batch data baru
pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Semarang", "Magelang"],
    "nama_menu": ["Cold Brew", "Americano"],
    "qty": [2, 3],
    "harga_satuan": [26000, 18000],
    "waktu_pesan": ["2026-11-11 10:03:45", "2026-11-11 10:03:55"],
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark menulis ke Parquet...")
time.sleep(8)

Batch 4 dikirim. Menunggu Spark menulis ke Parquet...


In [11]:
!ls -la tugas10_output

total 44
drwxr-xr-x 3 mazedev mazedev 4096 Nov 11 22:47 .
drwxrwxr-x 9 mazedev mazedev 4096 Nov 11 22:47 ..
-rw-r--r-- 1 mazedev mazedev 1857 Nov 11 22:47 part-00000-6501ed2a-64a1-4936-a1c0-8b829a5c3dd0-c000.snappy.parquet
-rw-r--r-- 1 mazedev mazedev   24 Nov 11 22:47 .part-00000-6501ed2a-64a1-4936-a1c0-8b829a5c3dd0-c000.snappy.parquet.crc
-rw-r--r-- 1 mazedev mazedev 1832 Nov 11 22:47 part-00001-bd12fd68-630f-4c97-b789-6405974b054c-c000.snappy.parquet
-rw-r--r-- 1 mazedev mazedev   24 Nov 11 22:47 .part-00001-bd12fd68-630f-4c97-b789-6405974b054c-c000.snappy.parquet.crc
-rw-r--r-- 1 mazedev mazedev 1847 Nov 11 22:47 part-00002-85bea42c-b377-411b-8116-6af75d6c18f6-c000.snappy.parquet
-rw-r--r-- 1 mazedev mazedev   24 Nov 11 22:47 .part-00002-85bea42c-b377-411b-8116-6af75d6c18f6-c000.snappy.parquet.crc
-rw-r--r-- 1 mazedev mazedev 1759 Nov 11 22:47 part-00003-05eefd4b-31a5-4272-abed-a71f208c8202-c000.snappy.parquet
-rw-r--r-- 1 mazedev mazedev   24 Nov 11 22:47 .part-00003-05eefd4b-31a5

In [12]:
# Membaca kembali hasil Parquet sebagai DataFrame batch biasa
df_parquet = spark.read.parquet("tugas10_output")
df_parquet.orderBy("pesanan_id").show(truncate=False)

+----------+-----------+-------------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu          |qty|harga_satuan|waktu_pesan        |
+----------+-----------+-------------------+---+------------+-------------------+
|P001      |Magelang   |Kopi Susu Gula Aren|2  |22000       |2026-11-11 10:00:15|
|P002      |Yogyakarta |Americano          |1  |18000       |2026-11-11 10:00:40|
|P003      |Magelang   |Matcha Latte       |3  |25000       |2026-11-11 10:01:05|
|P004      |Yogyakarta |Espresso           |2  |15000       |2026-11-11 10:01:30|
|P005      |Solo       |Cappuccino         |2  |24000       |2026-11-11 10:01:45|
|P006      |Magelang   |Kopi Tubruk        |1  |12000       |2026-11-11 10:01:55|
|P007      |Solo       |Latte              |3  |23000       |2026-11-11 10:02:10|
|P008      |Semarang   |Kopi Susu Gula Aren|4  |22000       |2026-11-11 10:02:35|
|P009      |Yogyakarta |Caramel Macchiato  |2  |28000       |2026-11-11 10:03:15|
|P010      |Sema

---
## Penutup — Menghentikan Seluruh Streaming Query

In [13]:
query.stop()
query2.stop()
query3.stop()
spark.stop()
print("Seluruh streaming query dan SparkSession berhasil dihentikan.")

Seluruh streaming query dan SparkSession berhasil dihentikan.
